# Preprocess and Upload Public Disclosure Data

## Load Libraries and Config Variables

In [ ]:
# import libraries
import collections
import copy
import json
import math
import os
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from lib.helpers import chunks, map_months_to_date
from pyseed.seed_client import SeedClient

curdir = Path(os.curdir).absolute().parent
seed_config = curdir / Path("seed-config-dev.json")
org_name = "Public Disclosure Example 1"
data_dir = curdir / "data"

client = SeedClient(
    None,
    connection_config_filepath=seed_config,
)

curdir = Path(os.curdir).absolute()
data_dir = curdir / "data" / "public_disclosure"
print(f"Data directory: {data_dir}")

output_to_upload_dir = data_dir / "to_upload"
print(f"Upload directory: {output_to_upload_dir}")
output_to_upload_dir.mkdir(parents=True, exist_ok=True)

# Allow autoreload as we develop potential libraries
%load_ext autoreload
%autoreload 2


warnings.filterwarnings("ignore")

In [ ]:
# Process the BEPS and Benchmarking Data from Washington DC

# Downloaded the BEPS data from
# https://opendata.dc.gov/datasets/DCGIS::building-energy-performance/explore
df_beps = pd.read_csv(data_dir / "Building_Energy_Performance.csv")

# All data are downloaded for benchmarking from
# https://opendata.dc.gov/datasets/DCGIS::building-energy-benchmarking/explore
# Download the file, open and save as an excel file. Add a filter and sort by
# PM Property ID. Spot check the data.
df_all = pd.read_excel(data_dir / "Building_Energy_Benchmarking.xlsx")

# ubid_file = data_dir / 'Unique_Building_Identifier.geojson'
# if not ubid_file.exists():
#     print("UBID file not found... did you extract the ZIP file?")

In [ ]:
display(df_all)

In [ ]:
# Summary of the beps and benchmarking data
reporting_years = np.sort(df_all["REPORTINGYEAR"].unique())
print(f"Reporting years for Benchmarking: {reporting_years}")
print(f"Reporting years for BEPS: {np.sort(df_beps['PROPERTY_BEPS_METRIC_YEAR'].unique())}")

# downselect the larger dataframe to only the reporting years of interest
# df = df[df['REPORTINGYEAR'].isin(reporting_years)]

# count of recorders for 2022
print(f"Total 2022 Benchmarking buildings: {len(df_all[df_all['REPORTINGYEAR'] == 2022])}")
print(f"Total 2019 BEPS Metric Year: {len(df_beps[df_beps['PROPERTY_BEPS_METRIC_YEAR'] == 2019])}")

# remove all the rows that don't have a PID (the unique property identifier for DC)
df_all = df_all[df_all["PID"].notna()]
df_beps = df_beps[df_beps["PID"].notna()]

# Append some of the BEPS data to the benchmarking data
columns_to_add = ["PID", "BEPS", "BEPS_METRIC_TYPE", "PROPERTY_BEPS_METRIC_YEAR"]
# there can be more than one row in df_all that need the df_beps data appended
df_all = df_all.merge(df_beps[columns_to_add], on="PID", how="left")

# create better property type groups
# Create a grouping of building types because there are many cases when the total
# buildings is not sufficient to notice trends
groups = {
    "Data Center": ["Data Center"],
    "Education": [
        "Adult Education",
        "College/University",
        "College/University (Campus-Level)",
        "K-12 School",
        "Other - Education",
        "Pre-school/Daycare",
    ],
    "Financial": ["Bank Branch", "Bank/Financial Institution", "Credit Union", "Financial Office"],
    "Grocery": ["Supermarket/Grocery Store"],
    "Hospital": [
        "Hospital (General Medical & Surgical)",
        "Hospital (General Medical and Surgical)",
        "Other - Specialty Hospital",
        "Other/Specialty Hospital",
    ],
    "Hospitality": ["Hotel", "Other - Lodging/Residential"],
    "Industrial": ["Manufacturing/Industrial Plant"],
    "Laboratory": ["Laboratory"],
    "Mall": ["Enclosed Mall", "Other - Mall", "Strip Mall"],
    "Medical Outpatient": [
        "Ambulatory Surgical Center",
        "Outpatient Rehabilitation/Physical Therapy",
        "Urgent Care/Clinic/Other Outpatient",
    ],
    "Multifamily": [
        "Multifamily Housing",
        "Residence Hall/Dormitory",
        "Residential Care Facility",
        "Senior Care Community",
        "Senior Living Community",
    ],
    "Office": ["Medical Office", "Office", "Other - Office", "Veterinary Office"],
    "Other": [
        "Mixed Use Property",
        "Other",
        "Other -Utility",
        "Parking",
        "Repair Services (Vehicle, Shoe, Locksmith, etc.)",
        "Other - Utility",
    ],
    "Public Assembly": [
        "Convention Center",
        "Movie Theater",
        "Other - Entertainment/Public Assembly",
        "Social/Meeting Hall",
        "Stadium (Closed)",
        "Stadium (Open)",
    ],
    "Public Services": [
        "Courthouse",
        "Fire Station",
        "Other - Public Services",
        "Police Station",
        "Post Office",
        "Prison/Incarceration",
        "Service (Vehicle Repair/Service, Postal Service)",
    ],
    "Recreation": [
        "Fitness Center/Health Club/Gym",
        "Ice/Curling Rink",
        "Indoor Arena",
        "Indoor Swimming Pool",
        "Library",
        "Museum",
        "Other - Recreation",
        "Performing Arts",
        "Recreation",
        "Swimming Pool",
    ],
    "Religious": ["Worship Facility"],
    "Restaurant": ["Bar/Nightclub", "Food Sales", "Food Sales & Service", "Food Service", "Other - Restaurant/Bar", "Restaurant"],
    "Retail": [
        "Other - Services",
        "Retail",
        "Retail Store",
        "Repair Services (Vehicle, Shoe, Locksmith, etc)",
        "Wholesale Club/Supercenter",
    ],
    "Single Family": ["Single Family Home"],
    "Treatment Plants": ["Drinking Water Treatment & Distribution", "Wastewater Treatment Plant", "Water Treatment Plant"],
    "Warehouse": [
        "Distribution Center",
        "Non-Refrigerated Warehouse",
        "Refrigerated Warehouse",
        "Self-Storage Facility",
        "Warehouse (Refrigerated)",
        "Warehouse (Unrefrigerated)",
    ],
}
# remove rows that have nan primary property types
df_all = df_all[df_all["PRIMARYPROPERTYTYPE_SELFSELECT"].notna()]
building_types = df_all["PRIMARYPROPERTYTYPE_SELFSELECT"].unique()
print(f"There are {len(building_types)} building types")

# check which properties we are missing -- it is the difference between all the buildings in the groups above
# and the buildings in common_building_types
missing = list(set(building_types) - set([item for sublist in groups.values() for item in sublist]))  # noqa
print(f"Missing building types: {missing}")
# check if there are any values that are not in the common_building_types
missing = [item for item in missing if item not in building_types]
print(f"Should not be included building types: {missing}")
# check for duplicates in the values of the groups
duplicates = [item for item, count in collections.Counter([item for sublist in groups.values() for item in sublist]).items() if count > 1]
print(f"Duplicate building types: {duplicates}")

# now add the groups to the df_all dataframe
df_all["Property Type Grouped"] = df_all["PRIMARYPROPERTYTYPE_SELFSELECT"].apply(
    lambda x: [k for k, v in groups.items() if x in v][0] if x in [item for sublist in groups.values() for item in sublist] else "Other"  # noqa
)

## Extract Monthly Meter Data from Benchmarking File

In [ ]:
# Generate an updated set in Excel with meters broken out

# df.columns.tolist()
ng_columns = [
    "NATURALGAS_KBTU_JANUARY",
    "NATURALGAS_KBTU_FEBRUARY",
    "NATURALGAS_KBTU_MARCH",
    "NATURALGAS_KBTU_APRIL",
    "NATURALGAS_KBTU_MAY",
    "NATURALGAS_KBTU_JUNE",
    "NATURALGAS_KBTU_JULY",
    "NATURALGAS_KBTU_AUGUST",
    "NATURALGAS_KBTU_SEPTEMBER",
    "NATURALGAS_KBTU_OCTOBER",
    "NATURALGAS_KBTU_NOVEMBER",
    "NATURALGAS_KBTU_DECEMBER",
]
elec_columns = [
    "ELECTRICITYUSE_KBTU_JANUARY",
    "ELECTRICITYUSE_KBTU_FEBRUARY",
    "ELECTRICITYUSE_KBTU_MARCH",
    "ELECTRICITYUSE_KBTU_APRIL",
    "ELECTRICITYUSE_KBTU_MAY",
    "ELECTRICITYUSE_KBTU_JUNE",
    "ELECTRICITYUSE_KBTU_JULY",
    "ELECTRICITYUSE_KBTU_AUGUST",
    "ELECTRICITYUSE_KBTU_SEPTEMBER",
    "ELECTRICITYUSE_KBTU_OCTOBER",
    "ELECTRICITYUSE_KBTU_NOVEMBER",
    "ELECTRICITYUSE_KBTU_DECEMBER",
]

other_energy_columns = [
    "DISTRCHILLEDWATER_KBTU",
    "DISTRHOTWATER_KBTU",
    "DISTRSTEAM_KBTU",
    "NATURALGASUSE_THERMS",
    "FUELOILANDDIESELFUELUSEKBTU",
    "ELECTRICITYUSE_RENEWABLE_KWH",
    "ELECTRICITYUSE_GRID_KWH",
]

annual_columns = other_energy_columns + [  # noqa
    "ENERGYSTARSCORE",
    "SITEEUI_KBTU_FT",
    "WEATHERNORMALZEDSITEEUI_KBTUFT",
    "SOURCEEUI_KBTU_FT",
    "WEATHERNORMALZEDSOUREUI_KBTUFT",
    "TOTGHGEMISSIONS_METRICTONSCO2E",
    "TOTGHGEMISSINTENSITY_KGCO2EFT",
    "WATERSCORE_MFPROPERTIES",
    "WATERUSE_ALLWATERSOURCES_KGAL",
]


# remove columns from all data
if "UBID" in df_all.columns:
    df_all = df_all.drop(columns=["UBID"])  # will be added with another dataset

# Move the BEPS columns to a new column based ont he BPS_METRIC_TYPE column. This will allow for easier tracking of dual types of BPS, and allow for units.
# find only where BEPS_METRIC_TYPE is ENERGYSTARSCORE and read in the BEPS value
df_all["BEPS_ENERGYSTAR"] = df_all[df_all["BEPS_METRIC_TYPE"] == "ENERGYSTARSCORE"]["BEPS"]
df_all["BEPS_SOURCEEUI"] = df_all[df_all["BEPS_METRIC_TYPE"] == "WEATHERNORMALZEDSOUREUI_KBTUFT"]["BEPS"]
# Now drop the other column (BEPS) to avoid confusion.
df_all = df_all.drop(columns=["BEPS"])

# Create pivot with ID and Electricity
df_all_energy = df_all[["PID", "REPORTINGYEAR"] + ng_columns + elec_columns + other_energy_columns]  # noqa
df_all_energy = df_all_energy.fillna(0)

# create a new dataframe without the monthly energy, but keep in the district energy data (which are annualized)
df_all_less_energy = df_all.drop(columns=ng_columns + elec_columns)

# Uncomment to help create the mapping template
# for c in df_all_less_energy.columns:
# print(f"{c},,PropertyState,{c}")

for year in np.sort(df_all_less_energy["REPORTINGYEAR"].unique()):
    # if year != 2019:
    # break

    print(f"Saving data for {year}...")
    df_all_less_energy[df_all_less_energy["REPORTINGYEAR"] == year].to_excel(
        output_to_upload_dir / f"dc_data_no_meters_{year}.xlsx", index=False
    )

# save off non-annual changing data for 2022 to 2024, but start from 2021 as the base year.
for year in [2023, 2024]:
    print(f"Saving future data for {year}...")
    df_outyears = df_all_less_energy[df_all_less_energy["REPORTINGYEAR"] == 2022]
    df_outyears["REPORTINGYEAR"] = year
    df_outyears = df_outyears.drop(columns=annual_columns)
    df_outyears.to_excel(output_to_upload_dir / f"dc_data_no_meters_{year}.xlsx", index=False)

In [ ]:
# now save the meter data in a json format, by PID
# {
#   <PID>: {
#       <meter_type>: {
#           meter_type: 'electricity',
#           units: 'kBtu',
#           source_unit: "Wh (Watt-hours)",
#           conversion_factor: 0.00341,
#           data: [
#               {
#                   "start_time": values[0],
#                   "end_time": values[1],
#                   "reading": values[2],
#                },
#                ...
#           ]
#       },
#       <meter_id2>: {
#          ...
# }
#   },
meter_data = {}
for year in np.sort(df_all_less_energy["REPORTINGYEAR"].unique()):
    for index, row in df_all_energy[df_all_energy["REPORTINGYEAR"] == year].iterrows():
        # if index > 10:
        # break

        if row["PID"] not in meter_data:
            meter_data[row["PID"]] = {}
        for meter in [
            "electricity",
            "natural_gas",
            "district_chilled_water",
            "district_hot_water",
            "district_steam",
            "fuel_oil_and_diesel",
        ]:
            # print(f'Creating meter data for {year} - {meter}...')
            # Meter type are the values defined in SEED's meter names
            if meter not in meter_data[row["PID"]]:
                meter_data[row["PID"]][meter] = {}
            meter_data[row["PID"]][meter]["source"] = "Manual Entry"
            meter_data[row["PID"]][meter]["source_id"] = row["PID"]
            if "first_non_zero_data_year" not in meter_data[row["PID"]][meter]:
                meter_data[row["PID"]][meter]["first_non_zero_data_year"] = None
            if "data" not in meter_data[row["PID"]][meter]:
                meter_data[row["PID"]][meter]["data"] = []

            if meter == "electricity":
                meter_data[row["PID"]][meter]["meter_type"] = "Electric - Grid"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                meter_data[row["PID"]][meter]["data"] += map_months_to_date(year, row[elec_columns].to_dict())
            elif meter == "natural_gas":
                meter_data[row["PID"]][meter]["meter_type"] = "Natural Gas"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                meter_data[row["PID"]][meter]["data"] += map_months_to_date(year, row[ng_columns].to_dict())
            elif meter == "fuel_oil_and_diesel":
                meter_data[row["PID"]][meter]["meter_type"] = "Custom Meter"
                meter_data[row["PID"]][meter]["alias"] = "Fuel Oil and Diesel"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                value = row["FUELOILANDDIESELFUELUSEKBTU"]
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is None and value != 0:
                    meter_data[row["PID"]][meter]["first_non_zero_data_year"] = int(year)
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is not None:
                    meter_data[row["PID"]][meter]["data"] += [
                        {
                            "start_time": f"{year}-01-01",
                            "end_time": f"{year}-12-31",
                            "reading": value,
                            "source_unit": "kBtu (Thousand BTU)",
                            "conversion_factor": 1,
                        }
                    ]
            elif meter == "district_chilled_water":
                meter_data[row["PID"]][meter]["meter_type"] = "District Chilled Water"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                value = row["DISTRCHILLEDWATER_KBTU"]
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is None and value != 0:
                    meter_data[row["PID"]][meter]["first_non_zero_data_year"] = int(year)
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is not None:
                    meter_data[row["PID"]][meter]["data"] += [
                        {
                            "start_time": f"{year}-01-01",
                            "end_time": f"{year}-12-31",
                            "reading": value,
                            "source_unit": "kBtu (Thousand BTU)",
                            "conversion_factor": 1,
                        }
                    ]
            elif meter == "district_hot_water":
                meter_data[row["PID"]][meter]["meter_type"] = "District Hot Water"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                value = row["DISTRHOTWATER_KBTU"]
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is None and value != 0:
                    meter_data[row["PID"]][meter]["first_non_zero_data_year"] = int(year)
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is not None:
                    meter_data[row["PID"]][meter]["data"] += [
                        {
                            "start_time": f"{year}-01-01",
                            "end_time": f"{year}-12-31",
                            "reading": value,
                            "source_unit": "kBtu (Thousand BTU)",
                            "conversion_factor": 1,
                        }
                    ]
            elif meter == "district_steam":
                meter_data[row["PID"]][meter]["meter_type"] = "District Steam"
                meter_data[row["PID"]][meter]["source_units"] = "kBtu"
                meter_data[row["PID"]][meter]["conversion_factor"] = 1
                value = row["DISTRSTEAM_KBTU"]
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is None and value != 0:
                    meter_data[row["PID"]][meter]["first_non_zero_data_year"] = int(year)
                if meter_data[row["PID"]][meter]["first_non_zero_data_year"] is not None:
                    meter_data[row["PID"]][meter]["data"] += [
                        {
                            "start_time": f"{year}-01-01",
                            "end_time": f"{year}-12-31",
                            "reading": value,
                            "source_unit": "kBtu (Thousand BTU)",
                            "conversion_factor": 1,
                        }
                    ]

# Break up the file into multiple files to make processing/saving easier
# with open(output_to_upload_dir / f"dc_meters_data_all.json", 'w') as outfile:
# json.dump(meter_data, outfile, indent=2)
i = 0
for item in chunks(meter_data, math.ceil(len(meter_data) / 10)):
    with open(output_to_upload_dir / f"dc_meters_data_{i}.json", "w") as outfile:
        json.dump(item, outfile, indent=2)
    i += 1  # noqa

## Upload the Building Data and Meters

In [ ]:
client.get_org_by_name(org_name, set_org_id=True)

client_info = client.instance_information()
print("You are connecting to the following instance:")
print(f"\tHost: {client_info['host']}")
print(f"\tVersion: {client_info['version']}")
print(f"\tSHA: {client_info['sha']}")
print(f"\tUsername: {client_info['username']}")

In [ ]:
# get the rang of dates in the data directory
files = list((data_dir / "to_upload").glob("dc_data_no_meters_*.xlsx"))
files.sort()
reporting_years = [int(f.stem[-4:]) for f in files]
print(reporting_years)

In [ ]:
# import DC building data in `to_upload` directory, but just do 2021 and 2022 for now
for year in reporting_years:
    if year not in [2021, 2022]:
        continue

    # verify that the file exists
    file = data_dir / "to_upload" / f"dc_data_no_meters_{year}.xlsx"
    if not file.exists():
        print(f"File {file} does not exist")
        continue

    # files to upload
    # continue

    # get the year, from the filename (ugh, silly!)
    print(f"Importing {file} for {year}")

    cycle = client.get_or_create_cycle(year, date(year, 1, 1), date(year, 12, 31), set_cycle_id=True)

    # upload the data to seed
    result = client.upload_and_match_datafile(
        "dc-building-data",
        file,
        "building data mappings",
        data_dir / "mappings-base-dc-data.csv",
        import_meters_if_exist=False,
    )

    print(result)

In [ ]:
# upload meter data

# create a "receipt" file that keeps track of which meters have been uploaded and
# check against the list to keep track of progress.
receipt_file = data_dir / "to_upload" / "dc_meters_receipt.json"
if not receipt_file.exists():
    # create a new file
    receipt_data = {}
elif os.stat(receipt_file).st_size == 0:
    receipt_data = {}
else:
    receipt_data = json.load(open(receipt_file))

# Create a column to store the meter_count, if not 0
client.create_extra_data_column(
    "meter_count",
    "Meter Count",
    "Property",
    "Number of meters manually calculated and pushed to SEED. Null and zero are equivalent.",
    "integer",
)

# use the most recent year for the cycle (it has the most data)
year = 2022
cycle = client.get_or_create_cycle(year, date(year, 1, 1), date(year, 12, 31), set_cycle_id=True)

# Find all the json files
files = list((data_dir / "to_upload").glob("dc_meters_data_*.json"))
files.sort()

for file in files:
    with open(file) as json_file:
        print(f"Processing {file}")
        data = json.load(json_file)
        for datum in data:
            # check if the data has the key and if so if the processed value is True
            if datum in receipt_data and receipt_data[datum]["processed"]:
                # data has already been processed, skipping
                continue
            else:
                # otherwise create the key
                receipt_data[datum] = {}
                receipt_data[datum]["processed"] = False
                # do not save until the state changes

            # For testing uncomment this line and verify that the data is being uploaded
            # correctly before trying to upload all of the data.
            # if not datum == 'PM1423516':
            # continue

            # Find the property
            property_obj = client.search_buildings(identifier_exact=datum)

            if len(property_obj) > 1:
                print(f"more than one obj found: {datum}")
                # save the meter_id to the receipt file
                receipt_data[datum]["processed"] = True
                receipt_data[datum]["status"] = "more_than_one_property"
                json.dump(receipt_data, open(receipt_file, "w"), indent=2)
                continue
            elif len(property_obj) == 0:
                # print(f'no property found for {datum}')
                # save the meter_id to the receipt file
                receipt_data[datum]["processed"] = True
                receipt_data[datum]["status"] = "no_property_found"
                json.dump(receipt_data, open(receipt_file, "w"), indent=2)
                continue

            property_view_id = property_obj[0]["id"]

            meter_count = 0
            for meter, value in data[datum].items():
                # Uncomment this section to re-upload (delete and reupload)

                # meter = client.get_meter(property_view_id,
                #     meter_type=value['meter_type'],
                #     source=value['source'],
                #     source_id=value['source_id'])
                # if meter:
                #     # right now if the meter exists, delete it so that we
                #     # don't have to fix the upsert method.
                #    result = client.delete_meter(property_view_id, meter['id'])

                # if there are meter data then create and upload, this will upsert
                # the data
                if value["data"]:
                    print(f"Data found -- processing {meter} for {property_view_id}")
                    meter_count += 1

                    seed_meter = client.get_or_create_meter(
                        property_view_id, meter_type=value["meter_type"], source=value["source"], source_id=value["source_id"]
                    )

                    payload = copy.deepcopy(value["data"])
                    # for item in payload:
                    # item['meter_id'] = meter['id']
                    result = client.upsert_meter_readings_bulk(property_view_id, seed_meter["id"], payload)

            # save the property's meter count (only if non-zero)
            if meter_count > 0:
                update_result = client.update_building(property_view_id, {"state": {"extra_data": {"meter_count": meter_count}}})

            # flag as processed
            receipt_data[datum]["processed"] = True
            receipt_data[datum]["meter_count"] = meter_count

            # save the meter_id to the receipt file
            json.dump(receipt_data, open(receipt_file, "w"), indent=2)

print("Finished")